# Harness Engineering Demo: Better Harness Beats Bigger Model

This notebook is built for a management-facing demo.

**Thesis:** a medium model with a good harness can beat a great model with a bad harness, because production quality depends on context, tools, validation, memory, safety gates, observability, and repair loops, not only raw model capability.

We will show the spectrum:

1. **Great model + bad harness**: bare prompt, no shared memory, no runbook enforcement.
2. **Medium model + hand-built harness**: explicit agents, shared memory, sensors, and scorecard.
3. **Medium model + SDK harness**: Strands-style abstraction for tools, hooks, memory, and multi-agent orchestration.
4. **Provider plug-and-play harness**: provider-specific harness lane, represented by DeepSeek adapter boundary.

The reliable part of the demo is deterministic. The live model section calls Ollama Cloud so management can see how this connects to real model backends.

## Demo Architecture

```text
Colab notebook
      ↓
Harness demo repo + Python SDKs
      ↓
Scenario: multi-agent incident response
      ↓
Scorecard: evidence, runbook, safety, memory, completeness
      ↓
Optional live calls to Ollama Cloud
```

Colab is the runtime. Ollama Cloud is the model backend.

## 1. Clone The Repo

Replace `REPO_URL` with your GitHub URL after pushing the project.

If you already uploaded this notebook into the cloned repo, skip this cell and `%cd` into the repo folder.

In [ ]:
# Replace this with your pushed GitHub repository URL.
REPO_URL = "https://github.com/YOUR_ORG/ollama-harness-engineering-demo.git"
REPO_DIR = "ollama-harness-engineering-demo"

from pathlib import Path
import os

# If we are not already inside the repo, clone it or move into an existing clone.
if not Path("pyproject.toml").exists():
    if Path(REPO_DIR).exists():
        os.chdir(REPO_DIR)
    else:
        if "YOUR_ORG" in REPO_URL:
            raise ValueError(
                "Replace REPO_URL with your GitHub repo URL, then rerun this cell. "
                "Example: https://github.com/my-org/ollama-harness-engineering-demo.git"
            )
        !git clone $REPO_URL
        os.chdir(REPO_DIR)

print("Current directory:", Path.cwd())
print("Project files:")
!ls -la

assert Path("requirements.txt").exists(), "requirements.txt not found. You are not inside the repo."
assert Path("pyproject.toml").exists(), "pyproject.toml not found. You are not inside the repo."

## 2. Install The Demo Dependencies

This happens inside Colab, so it does not depend on your office Mac allowing Python packages.

The important libraries are:

- `strands-agents`: SDK-level harness abstraction.
- `ollama`: direct calls to Ollama Cloud.
- `openai`: useful for OpenAI-compatible endpoints.
- `typer` and `rich`: CLI and readable scorecards.
- `pytest`: quick health check.

In [ ]:
from pathlib import Path

assert Path("requirements.txt").exists(), "Run the clone/%cd setup cell first. requirements.txt is missing here."
assert Path("pyproject.toml").exists(), "Run the clone/%cd setup cell first. pyproject.toml is missing here."

!pip install -r requirements.txt
!pip install -e .

## 3. Import The Repo Harness

From this point onward, Colab is only the presentation surface. The actual scenario and harness workflow are imported from the repo.

If imports fail, confirm that the clone/setup cell printed a directory ending in `ollama-harness-engineering-demo` and that `src/harness_demo` exists.

In [ ]:
from pathlib import Path
from pprint import pprint
import sys

repo_src = str(Path.cwd() / "src")
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)

assert Path("src/harness_demo").exists(), "Not in repo root or src/harness_demo is missing."

from rich.console import Console

from harness_demo.domain import Lane
from harness_demo.live import run_live_hand_built_lane, run_live_raw_lane
from harness_demo.reporting import print_comparison, print_result
from harness_demo.runners import RUNNERS
from harness_demo.scenarios import load_incident_scenario

console = Console(width=110)
scenario = load_incident_scenario("incident-response")

print("Loaded scenario:", scenario.id)
print("Scenario name:", scenario.name)

## 4. Show The Scenario Inputs

This is where the demo stops being prompt engineering.

The harness has separate artifacts that a production system would control:

- incident ticket
- logs
- runbook
- prior incident memory
- scoring contract

The live model does not get all of this as one giant magic prompt in the raw lane. The harnessed lane accesses it through controlled agent steps.

In [ ]:
print("INCIDENT")
pprint(scenario.incident)

print("\nEXPECTED QUALITY GATES")
pprint(scenario.expected)

In [ ]:
print("LOG TOOL DATA")
print(scenario.logs)

print("RUNBOOK TOOL DATA")
print(scenario.runbook)

print("PRIOR MEMORY TOOL DATA")
print(scenario.prior_memory)

## 5. Configure Ollama Cloud

Now we connect Colab to Ollama Cloud. Ollama Cloud is the model backend. The repo harness remains the application workflow.

Do not hardcode the API key in the notebook.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OLLAMA_API_KEY"):
    os.environ["OLLAMA_API_KEY"] = getpass("Enter OLLAMA_API_KEY: ")

print("OLLAMA_API_KEY configured:", bool(os.environ.get("OLLAMA_API_KEY")))

## 6. Choose Models

The intended comparison:

- **great model + bad harness**: one raw model call, no tools, no memory, no reviewer
- **medium model + good harness**: multiple controlled agent calls, tools, shared memory, sensors, reviewer, repair

Change these model names based on your Ollama Cloud subscription.

In [ ]:
RAW_STRONG_MODEL = "gpt-oss:120b"
HARNESS_MODEL = "gpt-oss:20b"

print("Raw strong model:", RAW_STRONG_MODEL)
print("Harnessed medium model:", HARNESS_MODEL)

# Live Demo: Great Model + Bad Harness

This makes **one** live Ollama Cloud call with only the incident prompt.

No log tool. No runbook tool. No prior memory. No reviewer. No repair loop.

The score is computed from the actual model output, not a static fake result.

In [ ]:
live_raw = run_live_raw_lane(scenario, model_name=RAW_STRONG_MODEL)
print_result(console, live_raw)

print("\nACTUAL MODEL OUTPUT")
print(live_raw.final_answer)

print("\nSCORED MEMORY EXTRACTED FROM ACTUAL OUTPUT")
pprint(live_raw.memory)

# Live Demo: Medium Model + Good Harness

This is the real harness demonstration.

The same medium model is called through several controlled agent steps:

1. log investigator agent gets only incident + logs
2. runbook agent gets only incident + runbook
3. memory agent gets only incident + prior memory
4. planner agent gets shared memory and required output contract
5. reviewer sensor checks forbidden actions and missing fields
6. repair agent runs only if reviewer finds issues

The score is computed from the shared memory produced by that workflow.

In [ ]:
live_harness = run_live_hand_built_lane(scenario, model_name=HARNESS_MODEL)
print_result(console, live_harness)

print("\nACTUAL AGENT OUTPUTS")
print(live_harness.final_answer)

## 7. Inspect Shared Memory From The Live Harness

This is the management moment. The improvement is not “a better prompt.” The improvement is that the system creates and scores a structured operational state.

In [ ]:
print("Incident facts")
pprint(live_harness.memory.incident_facts)

print("\nEvidence gathered")
for item in live_harness.memory.evidence:
    print("-", item)

print("\nRunbook steps selected")
for item in live_harness.memory.runbook_steps:
    print("-", item)

print("\nPrior lessons used")
for item in live_harness.memory.prior_lessons:
    print("-", item)

print("\nReviewer objections")
print(live_harness.memory.reviewer_objections or "None")

print("\nFinal plan")
pprint(live_harness.memory.final_plan)

## 8. Live Side-By-Side Scorecard

This is the claim we want to defend:

> A medium model with a good harness can compete with, and often beat, a stronger model with a weak harness.

Both rows below are produced from live Ollama Cloud calls.

In [ ]:
print_comparison(console, [live_raw, live_harness])

# Dry-Run Smoke Test, Not Demo Evidence

The deterministic runners are still useful, but only as a smoke test for the repo and notebook.

Do **not** present these static outputs as evidence that the harness works. Present the live section above.

In [ ]:
!harness-demo compare --scenario incident-response

In [ ]:
!python -m pytest -p no:cacheprovider

# SDK And Plug-And-Play Spectrum

After the live hand-built harness, explain where Strands fits:

```text
Hand-built harness:
  We explicitly coded tools, memory, sensors, reviewer, and repair.

Strands SDK harness:
  The same control ideas become SDK concepts: Agent, tools, hooks, sessions, memory, traces.

Provider harness:
  Provider-specific quirks become adapter behavior that the app can plug in.
```

This is the full business story:

> Harness engineering is moving from custom infrastructure toward reusable application-building blocks.

In [ ]:
strands_result = RUNNERS[Lane.STRANDS_SDK](scenario)
deepseek_result = RUNNERS[Lane.DEEPSEEK_PROVIDER](scenario)
print_comparison(console, [live_raw, live_harness, strands_result, deepseek_result])

# Closing Narrative

The medium model is not magically smarter. It wins when the harness gives it:

- controlled context
- tool boundaries
- shared memory
- policy/runbook grounding
- reviewer checks
- objective sensors
- repair loop
- repeatable scorecard

That is the difference between a chat answer and a production AI workflow.